Testing for importing the datasets and making sure the script works properly in and of itself before integration.

Test the datasets are loading properly

In [8]:
import sys
from pathlib import Path
import pe_dataset

MAX_BYTES = int(1.024 * 10**6)
malicious_path = Path("../Data/PE_Binaries/malicious_data")
benign_path = Path("../Data/PE_Binaries/benign_data")

data = pe_dataset.build_dataset_from_dir(benign_dir=benign_path, malware_dir=malicious_path, max_bytes=MAX_BYTES)


Getting some output of the dataset to see what the tensors are returning

In [9]:
import random

i = 0

while i < 200:
    index = random.randint(0, len(data))
    print(data.__getitem__(idx=index))

    i+=1

(tensor([120, 156, 236,  ..., 256, 256, 256]), tensor(1))
(tensor([120, 156, 237,  ..., 256, 256, 256]), tensor(1))
(tensor([120, 156, 236,  ...,  29,  72,  19]), tensor(1))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([120, 156, 236,  ..., 256, 256, 256]), tensor(1))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([ 77,  90, 144,  ..., 114, 111, 112]), tensor(0))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([ 77,  90, 144,  ..., 255,  72, 139]), tensor(0))
(tensor([120, 156, 237,  ..., 256, 256, 256]), tensor(1))
(tensor([120, 156, 236,  ..., 256, 256, 256]), tensor(1))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([120, 156, 204,  ..., 243, 124, 109]), tensor(1))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([ 77, 

For the the training loop I must:

1. Train the defender normally

2. Check if the warm up phase for the defender is exceeded

3. Discover what predictions the model made on a batch of malicious files

4. For the files that were correctly classified, place those into the attacker's input source

5. Train the attacker on the passed malicious binaries and run preturbations

6. Track whether or not each perturbation succeeds

7. Repeat 

First we need to split the dataset as usual

In [ ]:
import agents
import config

# This exists within pe_dataset.py, but I have it here to make things easier to test: changes made to pe_dataset.py require a reload
def split(dataset: pe_dataset.PEBinaryDataset, test_split=0.3, rand_state=42):
    from sklearn.model_selection import train_test_split

    data_labels = dataset.labels

    train_paths, test_paths, train_labels, test_labels = train_test_split(
        dataset.file_paths, data_labels, test_size=test_split,
        random_state=rand_state, stratify=data_labels)

    return train_paths, test_paths, train_labels, test_labels

NUM_EPOCHS = 20

(X_train_paths, X_test_paths, y_train, y_test) = split(data)

num_benign, num_mal = 0, 0

for label in y_train:
    if label == 1:
        num_benign+=1
    else:
        num_mal+=1

if num_mal == num_benign+1 or num_mal == num_benign-1:
    print("Benign/Malicious split is as balanced as it can be (there's always going to be 1 more benign or malicious)")
else:
    print(f"Benign/Malicious balance: 0->{num_benign}, 1->{num_mal}")


Benign/Malicious split is as balanced as it can be (there's always going to be 1 more benign or malicious)


In [22]:
from torch.utils.data import DataLoader
from Models.MalConv2 import LowMemConv, MalConv

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8

train_data = pe_dataset.PEBinaryDataset(file_paths=X_train_paths, labels=y_train, max_bytes=MAX_BYTES)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

test_data = pe_dataset.PEBinaryDataset(file_paths=X_test_paths, labels=y_test, max_bytes=MAX_BYTES)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=True)

lowmem = LowMemConv.LowMemConvBase(chunk_size=65535)

ModuleNotFoundError: No module named 'LowMemConv'